# RQ4 ONLY — HCLA Component Ablation

독립 실행용 노트북입니다.

- RQ4 only
- 50 cases × 4 conditions × 3 reps
- Checkpoint resume 지원
- 실행시간 기록
- CSV / PKL 저장
- Friedman + pairwise Wilcoxon + Holm 보정 포함

**주의:** `load_evidence_checkpoint()`, `qwen_generate()`, `judge_narrative()`, `judge_quality()`, `RQ345_OUTPUT_DIR` 등 공통 함수/변수는 앞선 공통 셀 또는 기존 노트북에서 정의되어 있어야 합니다.


In [ ]:
# ============================================================
# RQ4 ONLY - HCLA Component Ablation
# 독립 실행용
# ============================================================

import os
import time
import pickle
import json
import numpy as np
import pandas as pd

from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations


# ============================================================
# 0. RQ4 CONFIG
# ============================================================

RQ4_N_CASES = 50
RQ4_REPS = 3

RQ4_CONDITIONS = [
    "Full_HCLA",
    "Without_RoleSeparation",
    "Without_EvidenceConstraint",
    "Without_InteractiveRefinement",
]

RQ4_OUTPUT_DIR = os.path.join(
    RQ345_OUTPUT_DIR,
    "RQ4_Component_Ablation"
)

os.makedirs(RQ4_OUTPUT_DIR, exist_ok=True)


# ============================================================
# 1. RQ4 Prompt Definitions
# ============================================================

FULL_INST = """
You are the HCLA Justification component.

Use ONLY the supplied evidence.
Do not introduce external facts.
Internally verify every factual statement against the evidence.
State limitations when the supplied evidence is insufficient.

Write a concise expert-style explanation.
""".strip()


ROLE_PROMPTS = {
    "Without_RoleSeparation": """
You are an AI assistant.
Directly explain the prediction in one pass.
""".strip(),

    "Without_EvidenceConstraint": """
Explain the prediction using the supplied information
and any relevant reasoning you consider useful.
""".strip(),
}


# ============================================================
# 2. Groundedness Evaluation
# ============================================================

def rq4_grounded_rates(e, txt):
    judged = judge_narrative(e, txt)

    labels = [
        str(x.get("label", "")).upper()
        for x in judged.get("claims", [])
    ]

    n = len(labels)

    return {
        "ESR": labels.count("SUPPORTED") / n if n else np.nan,
        "UCR": labels.count("UNSUPPORTED") / n if n else np.nan,
        "CR": labels.count("CONTRADICTED") / n if n else np.nan,
        "claim_count": n,
    }


# ============================================================
# 3. RQ4 Generation
# ============================================================

def rq4_generate(e, condition):
    evidence = json.dumps(
        e,
        ensure_ascii=False,
        indent=2
    )

    if condition == "Full_HCLA":
        draft_prompt = (
            FULL_INST
            + "\n\nEvidence:\n"
            + evidence
            + "\n\nWrite a 2-4 sentence draft."
        )

        draft = qwen_generate(draft_prompt)

        refine_prompt = (
            "Review the draft ONLY against the supplied evidence.\n"
            "Remove unsupported, speculative, or overstated claims.\n"
            "Preserve useful evidence-based reasoning.\n"
            "Return ONLY the revised paragraph."
            "\n\nEvidence:\n"
            + evidence
            + "\n\nDraft:\n"
            + draft
        )

        return qwen_generate(
            refine_prompt,
            temperature=0.0
        )

    elif condition == "Without_InteractiveRefinement":
        prompt = (
            FULL_INST
            + "\n\nEvidence:\n"
            + evidence
            + "\n\nWrite a concise explanation."
        )

        return qwen_generate(prompt)

    elif condition in ROLE_PROMPTS:
        prompt = (
            ROLE_PROMPTS[condition]
            + "\n\nEvidence:\n"
            + evidence
            + "\n\nWrite a concise explanation."
        )

        return qwen_generate(prompt)

    else:
        raise ValueError(
            f"Unknown RQ4 condition: {condition}"
        )


# ============================================================
# 4. RQ4 RUN
# ============================================================

def run_rq4_only(
    evidence_list,
    reps=3,
    checkpoint_path=None
):
    rows = []
    completed = set()

    if checkpoint_path is not None and os.path.exists(checkpoint_path):
        try:
            old_df = pd.read_csv(checkpoint_path)

            if len(old_df) > 0:
                rows = old_df.to_dict("records")

                completed = set(
                    zip(
                        old_df["case_id"],
                        old_df["condition"],
                        old_df["rep"]
                    )
                )

                print(
                    f"♻️ RQ4 resume: "
                    f"{len(completed)} completed"
                )

        except Exception as ex:
            print(
                "⚠️ Existing checkpoint could not be loaded:",
                ex
            )

    total_jobs = (
        len(evidence_list)
        * len(RQ4_CONDITIONS)
        * reps
    )

    done = len(completed)

    print()
    print("=" * 70)
    print("RQ4 COMPONENT ABLATION")
    print("=" * 70)
    print("Cases       :", len(evidence_list))
    print("Conditions  :", len(RQ4_CONDITIONS))
    print("Repetitions :", reps)
    print("Total rows  :", total_jobs)
    print("Resume      :", done, "/", total_jobs)
    print("=" * 70)

    for case_idx, e in enumerate(evidence_list, start=1):
        for condition in RQ4_CONDITIONS:
            for rep in range(1, reps + 1):

                key = (
                    e["case_id"],
                    condition,
                    rep
                )

                if key in completed:
                    continue

                t0 = time.time()

                txt = rq4_generate(
                    e,
                    condition
                )

                generation_sec = (
                    time.time()
                    -
                    t0
                )

                t_ground = time.time()

                rates = rq4_grounded_rates(
                    e,
                    txt
                )

                grounded_sec = (
                    time.time()
                    -
                    t_ground
                )

                t_quality = time.time()

                quality = judge_quality(
                    e,
                    txt
                )

                quality_sec = (
                    time.time()
                    -
                    t_quality
                )

                total_sec = (
                    generation_sec
                    +
                    grounded_sec
                    +
                    quality_sec
                )

                row = {
                    "case_id": e["case_id"],
                    "true_label": e.get("true_label"),
                    "prediction": e.get("prediction"),
                    "anomaly_probability": e.get("anomaly_probability"),
                    "condition": condition,
                    "rep": rep,
                    "explanation": txt,
                    "word_count": len(str(txt).split()),
                    "generation_sec": generation_sec,
                    "grounded_judge_sec": grounded_sec,
                    "quality_judge_sec": quality_sec,
                    "total_sec": total_sec,
                    **rates,
                    **quality,
                }

                rows.append(row)
                completed.add(key)
                done += 1

                if checkpoint_path:
                    pd.DataFrame(rows).to_csv(
                        checkpoint_path,
                        index=False,
                        encoding="utf-8-sig"
                    )

                print(
                    f"✅ RQ4 "
                    f"{done}/{total_jobs} "
                    f"| case={e['case_id']} "
                    f"| {condition} "
                    f"| rep={rep} "
                    f"| {total_sec:.1f}s"
                )

    return pd.DataFrame(rows)


# ============================================================
# 5. Load Evidence
# ============================================================

print()
print("Loading evidence checkpoint...")

evidence_packages = load_evidence_checkpoint()

RQ4_N_CASES = min(
    RQ4_N_CASES,
    len(evidence_packages)
)

rq4_evidence = evidence_packages[:RQ4_N_CASES]

print("Evidence loaded:", len(evidence_packages))
print("RQ4 cases:", len(rq4_evidence))


# ============================================================
# 6. Experiment Plan
# ============================================================

estimated_rows = (
    RQ4_N_CASES
    * len(RQ4_CONDITIONS)
    * RQ4_REPS
)

print()
print("=" * 70)
print("RQ4 EXPERIMENT PLAN")
print("=" * 70)
print(f"Cases        : {RQ4_N_CASES}")
print(f"Conditions   : {len(RQ4_CONDITIONS)}")
print(f"Repetitions  : {RQ4_REPS}")
print(f"Result rows  : {estimated_rows}")
print("=" * 70)


# ============================================================
# 7. Checkpoint
# ============================================================

RQ4_CHECKPOINT = os.path.join(
    RQ4_OUTPUT_DIR,
    "rq4_checkpoint.csv"
)


# ============================================================
# 8. RUN
# ============================================================

rq4_start = time.time()

rq4_df = run_rq4_only(
    rq4_evidence,
    reps=RQ4_REPS,
    checkpoint_path=RQ4_CHECKPOINT
)

rq4_runtime_sec = (
    time.time()
    -
    rq4_start
)


# ============================================================
# 9. Add Experiment Metadata
# ============================================================

rq4_df["experiment_n_cases"] = RQ4_N_CASES
rq4_df["experiment_reps"] = RQ4_REPS
rq4_df["runtime_sec_total"] = rq4_runtime_sec


# ============================================================
# 10. SAVE FULL RESULTS
# ============================================================

RQ4_FULL_PATH = os.path.join(
    RQ4_OUTPUT_DIR,
    "rq4_ablation_full.csv"
)

rq4_df.to_csv(
    RQ4_FULL_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 11. SUMMARY
# ============================================================

RQ4_METRICS = [
    "ESR",
    "UCR",
    "CR",
    "clarity",
    "trust_support",
    "actionability",
]

available_metrics = [
    c for c in RQ4_METRICS
    if c in rq4_df.columns
]

rq4_summary = (
    rq4_df
    .groupby("condition")[available_metrics]
    .agg(["mean", "std"])
)

RQ4_SUMMARY_PATH = os.path.join(
    RQ4_OUTPUT_DIR,
    "rq4_ablation_summary.csv"
)

rq4_summary.to_csv(
    RQ4_SUMMARY_PATH,
    encoding="utf-8-sig"
)


# ============================================================
# 12. Runtime Summary
# ============================================================

runtime_info = pd.DataFrame([
    {
        "n_cases": RQ4_N_CASES,
        "reps": RQ4_REPS,
        "conditions": len(RQ4_CONDITIONS),
        "result_rows": len(rq4_df),
        "runtime_sec": rq4_runtime_sec,
        "runtime_min": rq4_runtime_sec / 60,
        "runtime_hour": rq4_runtime_sec / 3600,
        "mean_generation_sec": rq4_df["generation_sec"].mean(),
        "mean_grounded_judge_sec": rq4_df["grounded_judge_sec"].mean(),
        "mean_quality_judge_sec": rq4_df["quality_judge_sec"].mean(),
        "mean_total_sec_per_row": rq4_df["total_sec"].mean(),
    }
])

RQ4_RUNTIME_PATH = os.path.join(
    RQ4_OUTPUT_DIR,
    "rq4_runtime.csv"
)

runtime_info.to_csv(
    RQ4_RUNTIME_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 13. PKL CHECKPOINT
# ============================================================

RQ4_PKL_PATH = os.path.join(
    RQ4_OUTPUT_DIR,
    "rq4_results_qwen_llama.pkl"
)

with open(RQ4_PKL_PATH, "wb") as f:
    pickle.dump(
        {
            "results_df": rq4_df,
            "summary": rq4_summary,
            "runtime_info": runtime_info,
            "n_cases": RQ4_N_CASES,
            "reps": RQ4_REPS,
            "conditions": RQ4_CONDITIONS,
            "generator_model": globals().get(
                "QWEN_MODEL",
                "Qwen"
            ),
            "judge_model": globals().get(
                "LLAMA_JUDGE_MODEL",
                "Llama"
            ),
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )


# ============================================================
# 14. Statistical Significance
# Friedman + Pairwise Wilcoxon + Holm
# ============================================================

def holm_adjust(pvals):
    pvals = np.asarray(
        pvals,
        dtype=float
    )

    m = len(pvals)

    order = np.argsort(pvals)

    adjusted = np.empty(
        m,
        dtype=float
    )

    running = 0.0

    for rank, idx in enumerate(order):
        val = (
            (m - rank)
            *
            pvals[idx]
        )

        running = max(
            running,
            val
        )

        adjusted[idx] = min(
            1.0,
            running
        )

    return adjusted


stat_rows = []

for metric in available_metrics:

    pivot = rq4_df.pivot_table(
        index=[
            "case_id",
            "rep"
        ],
        columns="condition",
        values=metric,
        aggfunc="mean"
    )

    pivot = pivot.dropna(
        subset=RQ4_CONDITIONS
    )

    if len(pivot) >= 2:

        try:
            friedman_stat, friedman_p = (
                friedmanchisquare(
                    *[
                        pivot[c].values
                        for c in RQ4_CONDITIONS
                    ]
                )
            )

        except Exception:
            friedman_stat = np.nan
            friedman_p = np.nan

        stat_rows.append({
            "metric": metric,
            "test": "Friedman",
            "comparison": "All Conditions",
            "statistic": friedman_stat,
            "p_raw": friedman_p,
            "p_holm": np.nan,
            "n": len(pivot),
        })

    pair_results = []

    for c1, c2 in combinations(
        RQ4_CONDITIONS,
        2
    ):

        temp = pivot[
            [c1, c2]
        ].dropna()

        if len(temp) < 2:
            continue

        try:
            stat, p = wilcoxon(
                temp[c1],
                temp[c2]
            )

        except ValueError:
            stat = 0.0
            p = 1.0

        pair_results.append({
            "metric": metric,
            "test": "Wilcoxon",
            "comparison": f"{c1} vs {c2}",
            "statistic": stat,
            "p_raw": p,
            "n": len(temp),
        })

    if pair_results:

        adjusted = holm_adjust(
            [
                x["p_raw"]
                for x in pair_results
            ]
        )

        for result, adj_p in zip(
            pair_results,
            adjusted
        ):
            result["p_holm"] = adj_p
            stat_rows.append(result)


rq4_stats_df = pd.DataFrame(
    stat_rows
)

RQ4_STATS_PATH = os.path.join(
    RQ4_OUTPUT_DIR,
    "rq4_statistical_tests.csv"
)

rq4_stats_df.to_csv(
    RQ4_STATS_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. FINAL
# ============================================================

print()
print("=" * 70)
print("✅ RQ4 COMPLETE")
print("=" * 70)

print("Cases       :", RQ4_N_CASES)
print("Conditions  :", len(RQ4_CONDITIONS))
print("Reps        :", RQ4_REPS)
print("Rows        :", len(rq4_df))

print(
    "Runtime     :",
    f"{rq4_runtime_sec / 60:.2f} min"
)

print(
    "Runtime     :",
    f"{rq4_runtime_sec / 3600:.2f} hours"
)

print()
print("Full result :", RQ4_FULL_PATH)
print("Summary     :", RQ4_SUMMARY_PATH)
print("Stats       :", RQ4_STATS_PATH)
print("Runtime     :", RQ4_RUNTIME_PATH)
print("PKL         :", RQ4_PKL_PATH)

print("=" * 70)

try:
    display(
        rq4_summary.round(3)
    )

    display(
        rq4_stats_df
    )

except NameError:
    print(rq4_summary.round(3))
    print(rq4_stats_df)
